# 02 — Baseline & Classical Models

Train the Stage 1 baseline and compare Stage 2 classical models using the shared `src/` pipeline.

Models compared: Logistic Regression, Naive Bayes, SVM, Random Forest, XGBoost.

In [ ]:
import subprocess
import sys
from pathlib import Path

INSTALL_ON_REMOTE = False  # set True on Colab/Kaggle for first run

candidate = Path.cwd().resolve()
for path in (candidate, *candidate.parents):
    if (path / "pyproject.toml").exists():
        PROJECT_ROOT = path
        break
else:
    raise FileNotFoundError(
        "Could not find project root. Open this notebook from the repository."
    )

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

if INSTALL_ON_REMOTE:
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-q", ".[dev,notebook]"],
        cwd=PROJECT_ROOT,
    )

print(f"Project root: {PROJECT_ROOT}")

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from sklearn.metrics import ConfusionMatrixDisplay

from src.models.compare import compare_classical_models
from src.models.inference import load_baseline_artifacts, predict_text
from src.models.pipeline import train_tfidf_classifier
from src.models.classical import build_classical_model
from src.models.train import train_baseline
from utils.config import load_config

sns.set_theme(style="whitegrid")

BASELINE_CONFIG = PROJECT_ROOT / "configs/baseline.yaml"
CLASSICAL_CONFIG = PROJECT_ROOT / "configs/classical.yaml"

## Stage 1 — TF-IDF + Logistic Regression baseline

In [ ]:
baseline_result = train_baseline(BASELINE_CONFIG)
baseline_metrics = pd.Series(baseline_result["metrics"])
baseline_metrics

In [ ]:
vectorizer, classifier, config = load_baseline_artifacts(BASELINE_CONFIG)

demo_text = "Scientists confirm renewable energy breakthrough in peer-reviewed study."
prediction = predict_text(demo_text, vectorizer, classifier, config.get("preprocessing"))
prediction

## Stage 2 — Classical model comparison

In [ ]:
comparison = compare_classical_models(CLASSICAL_CONFIG)
comparison

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
sns.barplot(data=comparison, x="model", y="f1", ax=ax)
ax.set_title("Model comparison (F1 score)")
ax.set_xlabel("Model")
ax.set_ylabel("F1")
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()

## Confusion matrix — best model

In [ ]:
best_model = comparison.iloc[0]["model"]
classical_config = load_config(CLASSICAL_CONFIG)

model_config = {
    **classical_config.get("model", {}),
    **classical_config.get("model_configs", {}).get(best_model, {}),
    "name": best_model,
}
classifier = build_classical_model(best_model, model_config)

best_result = train_tfidf_classifier(
    config=classical_config,
    root=PROJECT_ROOT,
    classifier=classifier,
    model_name=best_model,
    output_dir=classical_config["output"]["model_dir"],
    save_artifacts=False,
)

cm = best_result["metrics"]["confusion_matrix"]
ConfusionMatrixDisplay(cm, display_labels=["real", "fake"]).plot(cmap="Blues")
plt.title(f"Confusion matrix — {best_model}")
plt.show()

metrics_path = PROJECT_ROOT / classical_config["output"]["model_dir"] / "model_comparison.csv"
print(f"Saved comparison table: {metrics_path}")